# N2 solution. Guided activity of Notebook 2, solved

**Predicting Flash Floods in Working Landscapes**, Module 1, Quantitative Precipitation Estimation (QPE)
CUAHSI Water Science Exchange 2026

This short notebook holds the completed activity of Notebook 2 (section 8), with its outputs. It rebuilds only what the activity needs, the 24 hourly fields of the three products and their basin averages over the Floyd River basin, so it runs on its own in about half a minute with the cache that ships with the repository. The full notebook with every section executed is `02_june2024_event_three_estimates_solution.ipynb`.

The task. It is 06:30 UTC on June 22, 2024, half an hour before the Flash Flood Emergency. Which rainfall estimate exists at that moment, and how far is it from the final amount.

## 0. Setup

In [1]:
import os, sys, subprocess, importlib.util
from pathlib import Path

REPO_URL = "https://github.com/AHWALab/Water-Science-Exchange-Workshop.git"
BRANCH = "module-1-qpe"

# Step 1. If the helper file is not next to this notebook, we are in Colab. Copy the branch (public, no account needed).
if not Path("qpe_tools.py").exists():
    if not Path("Water-Science-Exchange-Workshop").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "-b", BRANCH, REPO_URL, "Water-Science-Exchange-Workshop"], check=True)
    os.chdir("Water-Science-Exchange-Workshop/notebooks")
sys.path.insert(0, os.getcwd())

# Step 2. Install what is missing. Colab lacks the GRIB2 reader and the map library. About one minute the first time.
missing = [m for m in ["eccodes", "cfgrib", "cartopy", "s3fs", "h5netcdf"] if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
print("Working folder:", os.getcwd(), "| installed:", ", ".join(missing) if missing else "nothing, all packages present")

Working folder: /home/claude/m1/notebooks | installed: nothing, all packages present


## 1. What the activity needs from Notebook 2

The imports, the 24 hour window, the three hourly products clipped to the event box, the Floyd River basin and its hourly basin average rainfall (sections 1 and 7).

In [2]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import xarray as xr
from qpe_tools import *

EVENT_BOX = (-97.5, -94.5, 42.0, 44.5)                                   # west, east, south, north
HOURS = pd.date_range("2024-06-21 13:00", "2024-06-22 12:00", freq="h")   # files stamped with the END of each hour
PRODUCTS = ["RadarOnly", "Pass1", "Pass2"]

hourly = {p: read_mrms_series(p, HOURS, EVENT_BOX, verbose=False) for p in PRODUCTS}   # 24 fields per product
basin = nldi_basin("06600100")                                           # Floyd River at Alton, 694 square km
basin_rain = {p: basin_mean_series(hourly[p], basin) for p in PRODUCTS}  # basin mean, mm per hour, 24 values

for p in PRODUCTS:
    print(f"{LABELS[p]:22s} 24 h basin total {basin_rain[p].sum():5.1f} mm")

Radar only             24 h basin total 110.5 mm
Multi-sensor Pass 1    24 h basin total 116.2 mm
Multi-sensor Pass 2    24 h basin total 106.6 mm


## 2. The activity, solved

From Notebook 1, section 4, the files reach the archive about

- Radar only, within a few minutes. The hour ending 06:00 is in.
- Pass 1, about 20 minutes after the hour. The hour ending 06:00 is in.
- Pass 2, about one hour after the hour. The latest complete hour is 05:00.

In [3]:
#################### Your Code Goes Here #########################
# Activity, step 1. Define the decision time and the last available hour of each product.
DECISION = pd.Timestamp("2024-06-22 06:30")
last_hour = {"RadarOnly": pd.Timestamp("2024-06-22 06:00"),
             "Pass1": pd.Timestamp("2024-06-22 06:00"),
             "Pass2": pd.Timestamp("2024-06-22 05:00")}
###################################################################

In [4]:
#################### Your Code Goes Here #########################
# Activity, step 2. For each product, sum the basin mean rainfall over the hours available at the decision time.
#                   Use basin_rain[p] from section 7 and keep hours up to last_hour[p].
known = {}
for p in PRODUCTS:
    s = basin_rain[p]
    known[p] = s[s.index <= last_hour[p]].sum()
    print(f"{LABELS[p]:22s} known at {DECISION:%H:%M} UTC: {known[p]:5.1f} mm through hour ending {last_hour[p]:%H:%M}")
###################################################################

Radar only             known at 06:30 UTC: 103.2 mm through hour ending 06:00
Multi-sensor Pass 1    known at 06:30 UTC: 109.2 mm through hour ending 06:00
Multi-sensor Pass 2    known at 06:30 UTC:  91.5 mm through hour ending 05:00


In [5]:
#################### Your Code Goes Here #########################
# Activity, step 3. The answer in hindsight. Sum the final Pass 2 basin rainfall through 06:00
#                   and express each product's known total as a percent of it.
final = basin_rain["Pass2"][basin_rain["Pass2"].index <= pd.Timestamp("2024-06-22 06:00")].sum()
print(f"final Pass 2 through 06:00 UTC: {final:.1f} mm")
for p in PRODUCTS:
    print(f"{LABELS[p]:22s} {100 * known[p] / final:5.0f} percent of the final amount")
###################################################################

final Pass 2 through 06:00 UTC: 100.5 mm
Radar only               103 percent of the final amount
Multi-sensor Pass 1      109 percent of the final amount
Multi-sensor Pass 2       91 percent of the final amount


In [6]:
#################### Your Code Goes Here #########################
# Activity, step 4. Build the best estimate an analyst could assemble at 06:30,
#                   Pass 2 for the hours it covers, then Pass 1 for the hours after that.
#                   Hint: concatenate the two series after slicing them by time.
best = pd.concat([basin_rain["Pass2"][basin_rain["Pass2"].index <= last_hour["Pass2"]],
                  basin_rain["Pass1"][(basin_rain["Pass1"].index > last_hour["Pass2"]) &
                                      (basin_rain["Pass1"].index <= last_hour["Pass1"])]])
print(f"blended estimate at 06:30 UTC: {best.sum():.1f} mm, that is {100 * best.sum() / final:.0f} percent of the final amount")
###################################################################

blended estimate at 06:30 UTC: 101.4 mm, that is 101 percent of the final amount


Reading the result. At 06:30 UTC the radar only product already carried the whole basin rainfall, the gauge corrected products carried less because their last hour had not arrived, and the final Pass 2 amount, known only an hour later, was lower than the radar only amount. Latency and accuracy pull in opposite directions, and an analyst can blend the products to get the best of both.